In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-


# Deep Neural Networks

## Session 17a - Tensorflow

## Simple RNN - One feature


<img src='../../prasami_images/prasami_color_tutorials_small.png' width='400' alt="By Pramod Sharma : pramod.sharma@prasami.com" align="left"/>

In [ ]:
###-----------------
### Import Libraries
###-----------------

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

import tensorflow as tf

from utils.helper import fn_plot_tf_hist

In [ ]:
###----------------------
### Some basic parameters
###----------------------

inpDir = Path('../../input') # location where input data is stored
outDir = Path('../output') # location to store outputs

RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE) # Set Random Seed for reproducible results
tf.random.set_seed(RANDOM_STATE) # setting for Tensorflow as well

EPOCHS = 30  # number of cycles to run
ALPHA = 0.001  # learning rate
TEST_SIZE = 0.2 # What fraction we want to keep for testing
BATCH_SIZE = 32
TIME_STEPS = 24 # how many previous time steps to use as input variables to predict the next time period


# Set parameters for decoration of plots
params = {'legend.fontsize': 'medium',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'large',
          'axes.titlesize':'large',
          'xtick.labelsize':'medium',
          'ytick.labelsize':'medium'
         }
CMAP = plt.cm.coolwarm

plt.rcParams.update(params) # update rcParams

## Load Weather Data

In [ ]:
dataFilename = 'weatherHistory.csv'
data_df = pd.read_csv(inpDir / dataFilename)
data_df.head()

In [ ]:
data_df.shape

In [ ]:
data_df['datetime'] = pd.to_datetime(data_df['Formatted Date'], 
                                     utc=True)

data_df = data_df.sort_values('datetime').reset_index(drop=True)

In [ ]:
num_cols = ['Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)',
             'Visibility (km)', 'Pressure (millibars)']

fig, axes = plt.subplots(2,3, figsize=(15,8))

axes = axes.ravel()

for count, col in enumerate(num_cols):
    
    ax =axes[count]
    
    sns.histplot(data_df, x = col, ax = ax, bins = 50)

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(15,8))

axes = axes.ravel()

for count, col in enumerate(num_cols):

    sns.boxplot(y=col, data=data_df, ax = axes[count])

plt.tight_layout()
# Show the plot
plt.show()

In [ ]:
data_df[num_cols].corr()

In [ ]:
sns.heatmap(data_df[num_cols].corr(), annot=True, cmap='Dark2_r', linewidths = .2)
plt.show()

In [ ]:
temp_df = data_df[['datetime', 'Temperature (C)']]
temp_df.head()

In [ ]:
temp_df = temp_df.rename({'Temperature (C)': 'temp'}, axis=1)
temp_df.head()

In [ ]:
# comment/uncomment following lines if you want part or full dataset

# startDate = pd.to_datetime('2007-1-1', utc=True)
# endDate = pd.to_datetime('2008-1-1', utc=True)
# temp_df = temp_df[(temp_df['datetime']  >= startDate) & (temp_df['datetime']  < endDate)]

In [ ]:
temp_df.reset_index(drop=True, inplace = True)
temp_df.head()

In [ ]:
temp_df.shape

## Plotting samples

In [ ]:
fig, ax = plt.subplots(figsize = (15,12))
temp_df.plot(x='datetime', y='temp', style=".", ax = ax);
ax.grid()

In [ ]:
h_units = 100
input_shape = (1, 10000)

model = tf.keras.models.Sequential()

model.add(tf.keras.layers.Input(shape=input_shape))

model.add(tf.keras.layers.SimpleRNN(units = h_units, 
                                    activation = 'tanh' ))

model.add(tf.keras.layers.Dense(1, activation = 'linear' ))

In [ ]:
wax = model.get_weights()[0].shape
waa = model.get_weights()[1].shape
baa = model.get_weights()[2].shape
way = model.get_weights()[3].shape
bay = model.get_weights()[4].shape

print ('Shape of Matrix:')
print ('Wax = ', wax,'; Waa = ', waa, '; baa = ', baa,'; Way = ', way,'; bay = ', bay)

### Preparing Dataset

In [ ]:
# Create target indices (end of each window)
target_indices = np.arange(TIME_STEPS, len(temp_df), TIME_STEPS)
target_indices.shape

In [ ]:
4018*0.8//32

In [ ]:
# Create target dataframe
y_df = temp_df.iloc[target_indices].copy()


In [ ]:

# Create input dataframe (all data up to the last target)
X_df = temp_df.iloc[:len(y_df) * TIME_STEPS].copy()


In [ ]:

# Reshape X to (n_samples, time_step)
X = X_df['temp'].values.reshape(len(y_df), TIME_STEPS)


In [ ]:

# Use first (time_step-1) values as input
X = X[:, :TIME_STEPS - 1]  # Shape: (n_samples, time_step-1)


In [ ]:

# Extract target values
y = y_df['temp'].values


In [ ]:

# Split into train/test (temporal split, not random)
split_idx = int(len(X) * (1 - TEST_SIZE))
split_idx = split_idx//BATCH_SIZE*BATCH_SIZE # Adjust split index to be a multiple of batch size
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]


In [ ]:
X_train

### Scale inputs 

In [ ]:
# Fit the scaler ONLY on the training data to prevent data leakage
X_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1)).flatten()

## Converting to Datasets

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train_scaled,y_train_scaled))
test_ds = tf.data.Dataset.from_tensor_slices((X_test_scaled,y_test_scaled))

## Optimize for performance

train_ds = train_ds.cache()  # Cache first
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

test_ds = test_ds.cache()
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# Shuffle and batch the dataset
train_ds = train_ds.shuffle(buffer_size=X_train.shape[0]).batch(BATCH_SIZE)
test_ds = test_ds.shuffle(buffer_size=X_test.shape[0]).batch(BATCH_SIZE)

In [ ]:
h_units = 64
input_shape=(TIME_STEPS-1, 1 ) # we are using one feature only.

kernel_initializer = tf.keras.initializers.GlorotUniform(seed=RANDOM_STATE)

model = tf.keras.models.Sequential()

model.add(tf.keras.layers.Input(shape = input_shape))

model.add(tf.keras.layers.SimpleRNN(units = h_units,
                                    kernel_initializer=kernel_initializer,
                                    activation = 'tanh'))

model.add(tf.keras.layers.Dense(1, 
                                kernel_initializer=kernel_initializer,
                                activation = 'linear'))

model.compile(loss='mean_squared_error', optimizer='adam', 
              metrics=[tf.keras.metrics.RootMeanSquaredError()])

In [ ]:
model.summary()

In [ ]:
history = model.fit(train_ds,
                    epochs=EPOCHS, 
                    validation_data=test_ds,
                    batch_size= BATCH_SIZE, 
                    verbose=1)

In [ ]:
hist_df = pd.DataFrame(history.history)
hist_df = hist_df.rename({'root_mean_squared_error': 'rmse', 'val_root_mean_squared_error' : 'val_rmse'}, axis=1)


fn_plot_tf_hist(hist_df)

In [ ]:
# make predictions
y_train_pred_scaled = model.predict(X_train_scaled)
y_test_pred_scaled = model.predict(X_test_scaled)
y_pred_train = y_scaler.inverse_transform(y_train_pred_scaled.reshape(-1, 1))
y_pred_test = y_scaler.inverse_transform(y_test_pred_scaled.reshape(-1, 1))
y_pred =np.concatenate([y_pred_train, y_pred_test])

In [ ]:
y_pred.shape, y_pred_train.shape, y_pred_test.shape 

In [ ]:
y_df.shape

In [ ]:
res_df = y_df.copy()
res_df['pred'] = y_pred
res_df['datetime'] = res_df['datetime'].dt.date
res_df.head()

In [ ]:
res_df.tail()

In [ ]:
fig, ax = plt.subplots(figsize = (15,6))

res_df.plot(x='datetime', y=['temp','pred'], ax = ax);

ax.vlines(res_df.iloc[X_train.shape[0]]['datetime'], 
          res_df['temp'].min(), 
          res_df['temp'].max(), color = 'k', 
          linewidth=3.0, zorder=10, alpha =0.8)

ax.grid()